# Zepto E-commerce — SQL Solutions & Findings

This notebook analyzes product pricing, discounts, availability, stock value, unit economics and inventory weight. It reads from a MySQL `zepto` table and calculates findings from the actual rows.

**Prerequisites:** MySQL 8.0+ and populated `zepto` table. Set `MYSQL_HOST`, `MYSQL_PORT`, `MYSQL_USER`, `MYSQL_PASSWORD`, and `MYSQL_DATABASE`. The notebook assumes source prices are stored in paise, matching the original SQL comments. If your loaded file already stores rupees, change the conversion setting before running.

## 1. Connect and profile the source

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
required=["MYSQL_HOST","MYSQL_USER","MYSQL_PASSWORD","MYSQL_DATABASE"]
missing=[k for k in required if not os.getenv(k)]
if missing:
    raise EnvironmentError("Set MySQL environment variables: "+", ".join(missing))
url=(f"mysql+pymysql://{os.environ['MYSQL_USER']}:{os.environ['MYSQL_PASSWORD']}"
     f"@{os.environ['MYSQL_HOST']}:{os.getenv('MYSQL_PORT','3306')}/{os.environ['MYSQL_DATABASE']}")
engine=create_engine(url,pool_pre_ping=True)
def run_query(sql):
    return pd.read_sql_query(text(sql),engine)
source=run_query("""
SELECT COUNT(*) AS row_count,
       COUNT(DISTINCT sku_id) AS distinct_skus,
       COUNT(*)-COUNT(mrp) AS missing_mrp_rows,
       SUM(CASE WHEN mrp=0 THEN 1 ELSE 0 END) AS zero_mrp_rows,
       SUM(CASE WHEN discountedSellingPrice=0 THEN 1 ELSE 0 END) AS zero_selling_price_rows
FROM zepto
""")
display(source)
if source.empty or int(source.loc[0,"row_count"])==0:
    raise ValueError("zepto table is empty. Load the source dataset first.")

## 2. Build a non-destructive normalized view

This view converts paise to rupees and excludes zero/invalid MRP rows from pricing comparisons without deleting them from the source. It is safe to recreate. Do not run the conversion if the source values are already rupees.

In [ ]:
with engine.begin() as conn:
    conn.execute(text("DROP VIEW IF EXISTS zepto_analysis"))
    conn.execute(text("""
        CREATE VIEW zepto_analysis AS
        SELECT sku_id, category, name,
               mrp / 100.0 AS mrp_rupees,
               discountPercent,
               availableQuantity,
               discountedSellingPrice / 100.0 AS selling_price_rupees,
               weightInGms, outOfStock, quantity
        FROM zepto
        WHERE mrp > 0 AND discountedSellingPrice >= 0
    """))
print("Created zepto_analysis view. Source table remains unchanged.")

## 3. Q1 — Top ten discount offers

In [ ]:
q1=run_query("""
SELECT name, category, mrp_rupees, selling_price_rupees, discountPercent
FROM zepto_analysis
WHERE discountPercent IS NOT NULL
ORDER BY discountPercent DESC, name
LIMIT 10
""")
display(q1)
if not q1.empty:
    print(f"Finding: highest displayed discount is {q1.iloc[0]['discountPercent']}% for {q1.iloc[0]['name']}. Check final selling price and pack size before calling it best value.")

## 4. Q2 — High-MRP products out of stock

In [ ]:
q2=run_query("""
SELECT name, category, mrp_rupees, availableQuantity
FROM zepto_analysis
WHERE outOfStock=TRUE AND mrp_rupees>300
ORDER BY mrp_rupees DESC
""")
display(q2)
print(f"Finding: {len(q2):,} high-MRP product rows are flagged out of stock; this can help prioritise replenishment review, subject to inventory freshness.")

## 5. Q3 — Estimated inventory value by category

This calculation is **potential on-hand stock value**, not realized revenue: discounted price × available quantity.

In [ ]:
q3=run_query("""
SELECT category,
       SUM(selling_price_rupees*availableQuantity) AS estimated_inventory_value_rupees,
       SUM(availableQuantity) AS units_available,
       COUNT(*) AS sku_count
FROM zepto_analysis
GROUP BY category
ORDER BY estimated_inventory_value_rupees DESC
""")
display(q3)
if not q3.empty:
    r=q3.iloc[0]
    print(f"Finding: {r['category']} has the largest estimated inventory value (₹{r['estimated_inventory_value_rupees']:,.2f}) in the current snapshot.")

## 6. Q4 — Expensive products with discount below 10%

In [ ]:
q4=run_query("""
SELECT name, category, mrp_rupees, selling_price_rupees, discountPercent
FROM zepto_analysis
WHERE mrp_rupees>500 AND discountPercent<10
ORDER BY mrp_rupees DESC, discountPercent
""")
display(q4)
print(f"Finding: {len(q4):,} rows have MRP above ₹500 and discount below 10%; review competitor prices and product demand before changing price.")

## 7. Q5 — Categories with highest average discount

In [ ]:
q5=run_query("""
SELECT category, ROUND(AVG(discountPercent),2) AS avg_discount_pct,
       COUNT(*) AS sku_count
FROM zepto_analysis
GROUP BY category
ORDER BY avg_discount_pct DESC
LIMIT 5
""")
display(q5)
if not q5.empty:
    r=q5.iloc[0]
    print(f"Finding: {r['category']} has the highest average discount among the top five listed categories ({r['avg_discount_pct']}%). Compare margin and sales volume before interpreting discounts as effective.")

## 8. Q6 — Price per gram for products weighing at least 100g

In [ ]:
q6=run_query("""
SELECT name, category, weightInGms, selling_price_rupees,
       ROUND(selling_price_rupees/NULLIF(weightInGms,0),4) AS rupees_per_gram
FROM zepto_analysis
WHERE weightInGms>=100 AND weightInGms IS NOT NULL
ORDER BY rupees_per_gram, name
LIMIT 25
""")
display(q6)
print("Finding: lower rupees per gram can indicate better unit price within comparable products; compare category and pack composition before ranking across unrelated products.")

## 9. Q7 — Weight bands

In [ ]:
q7=run_query("""
SELECT CASE WHEN weightInGms<1000 THEN 'Low (<1kg)'
            WHEN weightInGms<5000 THEN 'Medium (1–<5kg)'
            ELSE 'Bulk (5kg+)' END AS weight_band,
       COUNT(*) AS sku_count,
       SUM(availableQuantity) AS units_available
FROM zepto_analysis
WHERE weightInGms IS NOT NULL
GROUP BY weight_band
ORDER BY sku_count DESC
""")
display(q7)
print("Finding: weight bands support assortment and handling analysis; confirm the cutoffs match the operational use case.")

## 10. Q8 — Inventory weight by category

In [ ]:
q8=run_query("""
SELECT category,
       SUM(weightInGms*availableQuantity) AS total_inventory_weight_gms,
       SUM(availableQuantity) AS units_available
FROM zepto_analysis
WHERE weightInGms IS NOT NULL AND availableQuantity IS NOT NULL
GROUP BY category
ORDER BY total_inventory_weight_gms DESC
""")
display(q8)
if not q8.empty:
    r=q8.iloc[0]
    print(f"Finding: {r['category']} has the highest estimated on-hand inventory weight ({r['total_inventory_weight_gms']:,.0f} grams). This can inform storage/handling review.")

## Executive summary and decision use
Use the returned tables to prioritise out-of-stock high-MRP products, compare discounting with unit value, and understand inventory value and weight by category. Validate source currency units, stock freshness, pack sizes and margin before taking pricing or replenishment actions. The analysis is descriptive and does not calculate realized revenue.